In [1]:
# общие импорты
# import numpy as np
# import pandas as pd

In [2]:
import duckdb as dd

conn = dd.connect()

In [3]:
users_db = "../data/raw/VK-LSVD/metadata/users_metadata.parquet"
items_db = "../data/raw/VK-LSVD/metadata/items_metadata.parquet"
bhv_db = "../data/raw/VK-LSVD/subsamples/up0.001_ip0.001/train/week_*.parquet"

# Eexploratory Data Analysis

## Данные о пользователях

In [9]:
df_meta = conn.execute(f"SUMMARIZE '{users_db}'").df()
print(df_meta)

               column_name column_type min        max  approx_unique  \
0                  user_id    UINTEGER  59  510119695       11420487   
1                      age    UTINYINT  18         70             50   
2                   gender    UTINYINT   1          2              2   
3                      geo    UTINYINT   0         79             77   
4  train_interactions_rank    UINTEGER   0    9999999        9008951   

                 avg                  std        q25        q50        q75  \
0  255064478.1068375    147255805.1144697  127567963  255328975  382414694   
1          36.724947   12.783831717831102         26         35         45   
2          1.5572987  0.49670603343426156          1          2          2   
3          64.099335    18.11393360839202         58         71         77   
4          4999999.5   2886751.4902856736    2500450    4993721    7501308   

      count  null_percentage  
0  10000000              0.0  
1  10000000              0.0  
2  10

Пропусков в таблице нет, количество значений в колонках везде соответствует количеству записей. 

Пользователи совершеннолетние, от 18 до 70 лет. 

Более 75% пользователей имеют собственный доход, про остальных непонятно. 

Более 50% принадлежат полу с id 2. 

Неожиданно, но больше 50% пользователей относятся к 6 регионам (можно предположить что это регионы, потому что их в россии где - то за 80, и учитывая обобщенный харрактер данных вполне возможна некоторая разница из за обобщения).

## Данные о фильмах/роликах

In [10]:
df_meta = conn.execute(f"SUMMARIZE '{items_db}'").df()
print(df_meta)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

               column_name column_type  min        max  approx_unique  \
0                  item_id    UINTEGER   38  608066952       17621978   
1                author_id    UINTEGER  558    1278886         509399   
2                 duration    UTINYINT    5        180            205   
3  train_interactions_rank    UINTEGER    0   19627600       24167530   

                 avg                 std        q25        q50        q75  \
0  304030904.9881613  175518069.10511392  152094855  303894155  456009946   
1  665026.1820389053   371168.8697332568     300957     660286    1008672   
2  34.11319136760524  29.769534356483707         12         24         50   
3          9813800.0   5666000.504785791    4899367    9811269   14710521   

      count  null_percentage  
0  19627601              0.0  
1  19627601              0.0  
2  19627601              0.0  
3  19627601              0.0  


Пропусков в данных нет. 

Четверь толиков имеет длительность менее 15 секунд, остальные - больше. 

Таблица связывает автора с роликами.

## Поведение пользователей

In [11]:
df_meta = conn.execute(f"DESCRIBE TABLE '{bhv_db}'").df()
print(df_meta)

        column_name column_type null   key default extra
0           user_id    UINTEGER  YES  None    None  None
1           item_id    UINTEGER  YES  None    None  None
2             place    UTINYINT  YES  None    None  None
3          platform    UTINYINT  YES  None    None  None
4             agent    UTINYINT  YES  None    None  None
5         timespent    UTINYINT  YES  None    None  None
6              like     BOOLEAN  YES  None    None  None
7           dislike     BOOLEAN  YES  None    None  None
8             share     BOOLEAN  YES  None    None  None
9          bookmark     BOOLEAN  YES  None    None  None
10  click_on_author     BOOLEAN  YES  None    None  None
11    open_comments     BOOLEAN  YES  None    None  None


In [7]:
df_meta = conn.execute(f"SUMMARIZE '{bhv_db}'").df()
df_meta

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,user_id,UINTEGER,20125,510091816,11146,253808535.00837898,146692900.64513347,127900224,252571775,380362943,47068641,0.0
1,item_id,UINTEGER,66761,608034240,18899,305236438.409698,175739131.8527498,151634898,306199543,456998666,47068641,0.0
2,place,UTINYINT,0,23,24,0.7952490703948729,0.6652516199543426,0,1,1,47068641,0.0
3,platform,UTINYINT,0,10,12,0.4765797253419745,0.8925757820847879,0,0,1,47068641,0.0
4,agent,UTINYINT,0,21,18,0.4195938438078125,0.6998881001759166,0,0,1,47068641,0.0
5,timespent,UTINYINT,1,255,265,15.48487934886414,23.06043319603746,2,6,19,47068641,0.0
6,like,BOOLEAN,false,true,2,NaN,NaN,NaN,NaN,NaN,47068641,0.0
7,dislike,BOOLEAN,false,true,2,NaN,NaN,NaN,NaN,NaN,47068641,0.0
8,share,BOOLEAN,false,true,2,NaN,NaN,NaN,NaN,NaN,47068641,0.0
9,bookmark,BOOLEAN,false,true,2,NaN,NaN,NaN,NaN,NaN,47068641,0.0


Пропусков в данных нет.

Таблица связывает действия пользователя с фильмом. 

При всем богатстве выбора place имеем только 2 источника, с которых народ переходил, если предположить что 0 является заполнителем, то и вовсе один.

Из всех возможных platform, задействованы тоже всего 2.

Агентов agent тоже всего 2.

Пожалуй самое важное поле, timespent, сколько пользователь смотрел конкретный ролик. Видим, как минимум четверь роликов попадают в первую перценталь и не выходят за пределы зачетных 5 секунд.

Далее идут поля пользовательских действий: активно понравилось или активно не понравилось, форварднул, сохранил, посмотрел данные автора, открыл коментарии.

### Чистка данных

In [8]:
# todo: убрать все записи о просмотрах роликов менее 5 секунд, если не было дополнительных действий пользователя